# 🏏 15 Years of IPL: What Makes Teams Successful?

## Objective

The goal of this project is to analyze 15 years of IPL data and identify the factors that contribute to team success. The analysis explores team performance, winning patterns, toss impact, player contributions, and scoring trends to uncover meaningful insights.

# IPL Data Analysis: Team Dominance, Toss Impact, Batting Evolution & Match Insights

## Business Problem

The Indian Premier League (IPL) is one of the world's most competitive T20 cricket leagues. This project analyzes historical IPL data to identify the factors behind team success, batting performance, venue influence, and match outcomes.

## Objectives

1. Identify the most successful IPL teams.
2. Analyze the impact of winning the toss.
3. Study batting evolution across seasons.
4. Find the greatest IPL performers.
5. Analyze venue influence on scoring.
6. Discover the most thrilling IPL matches.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
deliveries = pd.read_csv('../Data/deliveries.csv')
matches = pd.read_csv('../Data/matches.csv')



In [ ]:
deliveries.shape
deliveries.info()
deliveries.head()

In [ ]:

matches.shape
matches.info()
matches.head()


In [ ]:
matches.isnull().sum().sort_values(ascending= False)

In [ ]:
deliveries.isnull().sum().sort_values(ascending=False)

In [ ]:
sorted(matches['team1'].unique())
sorted(matches['team2'].unique())
sorted(matches['winner'].dropna().unique())

In [ ]:
team_rename = {
    'Delhi Daredevils' : 'Delhi Capitals',
    'Kings XI Punjab' : 'Punjab Kings',
    'Rising Pune Supergiant' : 'Rising Pune Supergiants',
    'Royal Challengers Bangalore':'Royal Challengers Bengaluru'


}

matches['team1'] = matches['team1'].replace(team_rename)
matches['team2'] = matches['team2'].replace(team_rename)
matches['winner'] = matches['winner'].replace(team_rename)
matches['toss_winner'] = matches['toss_winner'].replace(team_rename)
deliveries['batting_team'] = deliveries['batting_team'].replace(team_rename)






In [ ]:
sorted(matches['team1'].unique())

#  Team Dominance Analysis

This section analyzes the overall performance of IPL teams based on matches played, total wins, win percentage, and consistency across seasons.

In [ ]:
total_matches = matches['team1'].value_counts() + matches['team2'].value_counts()
matches_played = total_matches.reset_index()
matches_played.columns = ['Team', 'Matches_Played']
matches_played.head(10)

In [ ]:
# Most Sucessfull Team

wins = matches['winner'].value_counts().reset_index()
wins.columns = ['Team','Wins']
wins.head(10)

In [ ]:
team_stats = pd.merge(matches_played, wins, on = 'Team' )
team_stats.head(10)

In [ ]:
team_stats['Win_Percentage'] = (
    team_stats['Wins'] / team_stats['Matches_Played']
) * 100

team_stats.head(10)

In [ ]:
team_stats.sort_values(
    by='Win_Percentage',
    ascending=False
).head(10)

### Key Insight

Gujarat Titans have the highest win percentage (62.22%) among all IPL teams. However, they have played only 45 matches, which is significantly fewer than long-established franchises.

Among teams with over 200 matches played, Chennai Super Kings (57.98%) and Mumbai Indians (55.17%) have demonstrated the strongest long-term consistency and success.

In [ ]:
matches.groupby('season')['id'].count()

In [ ]:
matches.groupby(['season', 'winner'])['id'].count()

In [ ]:
season_wins = matches.groupby(['season','winner'])['id'].count().reset_index()

season_wins.columns = ['Season', 'Team','Wins' ]
season_wins.head(10)

In [ ]:
season_wins[season_wins['Team']==  'Chennai Super Kings']

In [ ]:
season_wins.groupby('Team')['Wins'].mean().sort_values(ascending=False)

In [ ]:
import os

os.makedirs('Images', exist_ok=True)

## Team Win Percentage

Win percentage provides a fair comparison between teams by accounting for the number of matches played.

In [ ]:


team_stats.sort_values(
    by='Win_Percentage',
    ascending=False
).plot(
    x='Team',
    y='Win_Percentage',
    kind='bar',
    figsize=(12,6)
)

plt.title('IPL Teams Ranked by Win Percentage')
plt.xlabel('Teams')
plt.ylabel('Win Percentage (%)')
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig('Images/team_win_percentage.png', dpi=300, bbox_inches = 'tight')

plt.show()

### Key Insight

- Gujarat Titans recorded the highest win percentage (62.22%).
- However, they have participated in significantly fewer matches than long-established franchises.
- Chennai Super Kings (57.98%) and Mumbai Indians (55.17%) have maintained elite win rates across more than 200 matches, demonstrating sustained excellence.

# Does Winning the Toss Actually Matter?

A common belief in cricket is that winning the toss provides a significant advantage. This section investigates whether toss winners actually win matches more often.

In [ ]:
(matches['toss_winner'] == matches['winner']).sum()

In [ ]:
toss_match_win_pct =(
    (matches['toss_winner'] == matches['winner']).sum()
    / len(matches)
) * 100

toss_match_win_pct

## Toss Winner vs Match Winner

A common belief among cricket fans is that winning the toss provides a significant advantage.

However, analysis of IPL matches shows that teams winning the toss also won the match only 50.6% of the time.

This suggests that winning the toss alone does not significantly influence match outcomes, and team performance remains the primary determinant of success.

In [ ]:
matches['toss_decision'].value_counts()

In [ ]:
bat_wins = matches[
    (matches['toss_decision'] == 'bat') &
    (matches['toss_winner']== matches['winner'])
].shape[0]

field_wins = matches[
    (matches['toss_decision'] == 'field') &
    (matches['toss_winner']== matches['winner'])

].shape[0]

print("Bat decision wins:", bat_wins)
print("Field decision wins:", field_wins)




In [ ]:
bat_total = (matches['toss_decision']== 'bat').sum()
field_total = (matches['toss_decision'] == 'field').sum()

bat_win_pct = (bat_wins / bat_total)* 100
field_win_pct = (field_wins / field_total)* 100

print("Bat First Success Rate:", round(bat_win_pct,2), "%")
print("Field First Success Rate:", round(field_win_pct,2), "%")

## Toss Impact Analysis

One of the most debated topics in cricket is whether winning the toss provides a significant advantage.

Analysis of IPL matches shows:

- Teams choosing to field first won 53.55% of matches.
- Teams choosing to bat first won 45.27% of matches.
- Captains preferred to field first in over 64% of matches.

### Key Insight

The data supports the popular belief that chasing offers a slight competitive advantage in IPL matches.

In [ ]:
toss_results = pd.DataFrame({
    'Decision':['Bat_First', 'Field_First'],
    'Success_Rate': [bat_win_pct, field_win_pct]

})

toss_results.plot(
    x = 'Decision',
    y = 'Success_Rate',
    kind = 'bar',
    figsize =(6,4)
)
plt.title('Success Rate By Toss Decision')
plt.xlabel("Toss Decision")
plt.ylabel('Win Percentge %')
plt.tight_layout()

plt.savefig('Images/toss_impact.png', dpi=300, bbox_inches='tight')

plt.show()

## IPL Batting Evolution

Cricket has changed significantly since the first IPL season.

This analysis investigates whether scoring patterns have increased over time, indicating a shift toward a more batting-friendly tournament.


- Has the average team score increased over the years?
- Are teams scoring more aggressively in recent seasons?
- Has IPL become a batting-dominated competition?

In [ ]:
deliveries.columns

In [ ]:
team_scores = deliveries.groupby(['match_id','inning']) ['total_runs'].sum().reset_index()
team_scores.head()

In [ ]:
team_scores = pd.merge(team_scores, matches[['id','season']],left_on = 'match_id', right_on='id')
team_scores.drop('id', axis = 1,inplace = True)
team_scores.head()

In [ ]:
avg_score_by_season = team_scores.groupby('season')['total_runs'].mean().round(2)
avg_score_by_season

In [ ]:
avg_score_by_season = avg_score_by_season.reset_index()
avg_score_by_season.head(10)


In [ ]:
avg_score_by_season.plot(
    x='season',
    y='total_runs',
    kind='line',
    figsize=(12,5),
    marker ='o',
    color= 'red'

)

plt.title('Average IPL Team Score by Season')
plt.xlabel('Season')
plt.ylabel('Average Runs Per Innings')
plt.xticks(rotation=45)
plt.grid(True)

plt.tight_layout()

plt.savefig('Images/avg_team_score_by_season.png', dpi = 500, bbox_inches ='tight' )
plt.show()


### Average Team Score by IPL Season

The average innings score was calculated for each IPL season to understand how scoring patterns have evolved over time.

This analysis helps identify whether the IPL has become more batting-friendly and whether teams are consistently posting higher totals in recent years.

### Key Insight

Average team scores have generally increased over the history of the IPL.

While scoring fluctuated between seasons, a strong upward trend is visible from 2022 onwards.

The 2024 season recorded the highest average innings score in IPL history, indicating a highly batting-friendly era driven by aggressive batting strategies, improved player skill levels, and modern T20 tactics.

In [ ]:
team_avg_score = deliveries.groupby('batting_team')['total_runs'].sum()

innings_count = deliveries.groupby(['match_id','inning','batting_team'])['total_runs'].sum().reset_index()

team_inning = innings_count.groupby('batting_team').size()

team_avg_score = (
    team_avg_score / team_inning
).sort_values(ascending=False)

team_avg_score.head(10)


### Top 10 IPL Teams by Average Score

This visualization compares the average runs scored per innings by the top-performing IPL teams.

Average score is calculated by dividing total runs scored by total innings played, providing a fair comparison across teams with different numbers of matches.

Higher average scores indicate stronger batting performances and greater scoring consistency.

In [ ]:
team_avg_score.head(10).plot(
    kind='bar',
    figsize=(10,5)
)

plt.title('Top 10 IPL Teams by Average Score')
plt.xlabel('Team')
plt.ylabel('Average Runs Per Innings')
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(
    'Images/top_10_team_avg_score.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

### Key Insight

Royal Challengers Bengaluru recorded the highest average innings score among IPL teams.

Newer franchises such as Gujarat Titans and Lucknow Super Giants also rank highly, reflecting the modern high-scoring nature of the IPL.

Traditional teams like Chennai Super Kings and Mumbai Indians remain among the strongest batting sides while maintaining performance over a much larger number of innings.

### Highest Team Totals in IPL History



In [ ]:
highest_scores = deliveries.groupby(['match_id','inning', 'batting_team'])['total_runs'].sum().reset_index()
highest_scores.sort_values(by ='total_runs',ascending = False).head(10)

In [ ]:
highest_scores = pd.merge(
    highest_scores,
    matches[['id','season']],
    left_on = 'match_id',
    right_on='id'

)
highest_scores.drop('id', axis=1,inplace=True)

highest_scores.sort_values(by = 'total_runs',ascending=False).head(10)

In [ ]:
season_highest_team = highest_scores.loc[
    highest_scores.groupby('season')['total_runs'].idxmax()
].reset_index()

season_highest_team.head(10)

In [ ]:
season_highest_team.plot(
    x='season',
    y='total_runs',
    kind='line',
    figsize=(12,6),
    marker ='o',
    color= 'red'

)
plt.title('Highest Team Score in Each IPL Season')
plt.xlabel('Season')
plt.ylabel('Runs')
plt.xticks(rotation=45)
plt.grid(True)

plt.tight_layout()

plt.savefig('Images/season_highest_team_score.png',dpi=500,bbox_inches='tight')

plt.show()


## Highest Team Score by Season

This visualization shows the highest team total recorded in each IPL season.

### Key Insight
- Team scores have generally increased over time.
- Recent IPL seasons have witnessed several record-breaking totals.
- The rise in scoring reflects aggressive batting approaches, improved power-hitting, and batting-friendly conditions.

In [ ]:
top_10_scores = highest_scores.sort_values(by = 'total_runs',ascending= False).head(10)

top_10_scores

# Create unique labels here
top_10_scores['team_score'] = (
    top_10_scores['batting_team']
    + ' ('
    + top_10_scores['total_runs'].astype(str)
    + ')'
)
top_10_scores

In [ ]:
plt.figure(figsize=(12,6))

bars = plt.barh(
    top_10_scores['team_score'],
    top_10_scores['total_runs'],
    color = 'green'
)

plt.title('Top 10 Highest Team Totals in IPL History')
plt.xlabel('RUNS')
plt.ylabel('TEAMS')

for bar in bars:
    plt.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height()/2,
        str(int(bar.get_width())),
        va='center'
    )

plt.gca().invert_yaxis()

plt.tight_layout()

plt.savefig(
    'Images/top_10_highest_team_totals.png',
    dpi=500,
    bbox_inches='tight'
)

plt.show()

### Top 10 Highest Team Totals in IPL History

- Sunrisers Hyderabad recorded the highest team total in IPL history with **287 runs**.
- Sunrisers Hyderabad appears multiple times among the highest IPL totals.
- Kolkata Knight Riders and Royal Challengers Bengaluru also feature prominently in the top-scoring innings list.
- Most of the highest totals have been recorded in recent IPL seasons, indicating the growing dominance of aggressive batting strategies.

In [ ]:
team_wins = matches['winner'].value_counts()

team_wins.head(10)

In [ ]:
top_10_wins = team_wins.head(10)

plt.figure(figsize=(12,6))

bars = plt.bar(
    top_10_wins.index,
    top_10_wins.values
)

plt.title('Top 10 Most Successful IPL Teams')
plt.xlabel('Teams')
plt.ylabel('Matches Won')

plt.xticks(rotation=45)

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 1,
        str(int(bar.get_height())),
        ha='center'
    )

plt.tight_layout()

plt.savefig(
    'Images/top_10_team_wins.png',
    dpi=500,
    bbox_inches='tight'
)

plt.show()

### Teams with Most Wins in IPL History

- Mumbai Indians and Chennai Super Kings are among the most successful franchises in IPL history.
- These teams have consistently performed well across multiple seasons.
- Kolkata Knight Riders and Royal Challengers Bengaluru also rank among the top teams by total wins.
- Match wins provide a strong indicator of long-term franchise success and consistency.

In [ ]:
top_batsmen=(deliveries.groupby('batter')['batsman_runs'].sum().sort_values(ascending=False))

top_batsmen.head(10)






In [ ]:
top_10_batsmen = top_batsmen.head(10)

plt.figure(figsize=(12,6))

bars = plt.barh(
    top_10_batsmen.index,
    top_10_batsmen.values,
    color='orange'
)

plt.title('Top 10 Run Scorers in IPL History')
plt.xlabel('Runs')
plt.ylabel('Batter')

for bar in bars:
    plt.text(
        bar.get_width()+20,
        bar.get_y()+bar.get_height()/2,
        str(int(bar.get_width())),
        va='center'
    )

plt.gca().invert_yaxis()

plt.tight_layout()

plt.savefig(
    'Images/top_10_run_scorers.png',
    dpi=500,
    bbox_inches='tight'
)

plt.show()

### Top Run Scorers in IPL History

- Virat Kohli is among the leading run scorers in IPL history.
- Consistent batting performances over multiple seasons contribute significantly to a player's total runs.
- The top run scorers demonstrate remarkable longevity and consistency in the tournament.
- Run aggregates help identify the most successful batters across IPL seasons.

In [ ]:
top_bowlers = (
    deliveries[deliveries['is_wicket'] == 1]
    .groupby('bowler')['is_wicket']
    .count()
    .sort_values(ascending=False)
)

top_bowlers.head(10)

In [ ]:
top_10_bowlers = top_bowlers.head(10)

plt.figure(figsize=(12,6))

bars = plt.barh(
    top_10_bowlers.index,
    top_10_bowlers.values,
    color='purple'
)

plt.title('Top 10 Wicket Takers in IPL History')
plt.xlabel('Wickets')
plt.ylabel('Bowler')

for bar in bars:
    plt.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height()/2,
        str(int(bar.get_width())),
        va='center'
    )

plt.gca().invert_yaxis()

plt.tight_layout()

plt.savefig(
    'Images/top_10_wicket_takers.png',
    dpi=500,
    bbox_inches='tight'
)

plt.show()

### Top 10 Wicket Takers in IPL History

- Yuzvendra Chahal is among the leading wicket-takers in IPL history.
- Dwayne Bravo and Piyush Chawla also feature prominently in the top wicket-taking list.
- Consistent wicket-taking ability over multiple seasons is a key indicator of bowling success.
- The top wicket-takers have played crucial roles in restricting opposition scoring and winning matches for their teams.

In [ ]:
deliveries.columns

In [ ]:
individual_scores = (
    deliveries.groupby(['match_id', 'batter'])['batsman_runs']
    .sum()
    .reset_index()
)

top_10_individual = (
    individual_scores
    .sort_values('batsman_runs', ascending=False)
    .head(10)
)

top_10_individual

In [ ]:

top_10_individual = pd.merge(
    top_10_individual,
    matches[['id', 'season']],
    left_on='match_id',
    right_on='id'
)

top_10_individual

top_10_individual['label'] = (
    top_10_individual['batter']
    + ' (' +
    top_10_individual['batsman_runs'].astype(str)
    + ')'
)
top_10_individual


In [ ]:
plt.close('all')

fig, ax = plt.subplots(figsize=(12,6))

bars = ax.barh(
    top_10_individual['label'],
    top_10_individual['batsman_runs'],
    color='red'
)

ax.set_title('Top 10 Individual Scores in IPL History')
ax.set_xlabel('Runs')
ax.set_ylabel('Batter')

for bar in bars:
    ax.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height()/2,
        f'{int(bar.get_width())}',
        va='center'
    )

plt.tight_layout()
plt.savefig(
    'Images/Top 10 Individual Scores in IPL History.png',
    dpi=500,
    bbox_inches='tight'
)
plt.show()

- Y-axis = Bowler Name
- X-axis = Total Wickets
- Longer bar = More wickets
- Helps identify the most successful wicket-taking bowlers in IPL history

#  Venue Analysis

Venue conditions can significantly influence match outcomes and scoring patterns.

This section investigates:

- Which venues host the most IPL matches?
- Which venues are the highest-scoring grounds?
- Does venue influence team performance?

In [ ]:
venue_counts = matches['venue'].value_counts()

venue_counts.head(10)

In [ ]:
top_venues = venue_counts.head(10)

ax = top_venues.sort_values().plot(
    kind='barh',
    figsize=(12,6),
    color='purple'
)

plt.title('Top 10 IPL Venues by Matches Hosted')
plt.xlabel('Matches Hosted')
plt.ylabel('Venue')

for bar in ax.patches:
    plt.text(
        bar.get_width()+0.5,
        bar.get_y()+bar.get_height()/2,
        str(int(bar.get_width())),
        va='center'
    )

plt.tight_layout()
plt.savefig(
    'Images/top_10_wicket_takers.png',
    dpi=500,
    bbox_inches='tight'
)
plt.show()

### Key Findings

- Eden Gardens hosted the highest number of IPL matches.
- Wankhede Stadium and M Chinnaswamy Stadium are among the most frequently used venues.
- Historic IPL venues continue to dominate match hosting responsibilities.

In [ ]:
venue_data = deliveries.merge(
    matches[['id', 'venue']],
    left_on='match_id',
    right_on='id',
    how='left'
)

In [ ]:
venue_runs = venue_data.groupby('venue')['total_runs'].sum()

venue_matches = matches['venue'].value_counts()

avg_venue_score = (
    venue_runs / venue_matches
).sort_values(ascending=False)

avg_venue_score.head(10)

In [ ]:
top_10_venues = avg_venue_score.head(10).sort_values()

ax = top_10_venues.plot(
    kind='barh',
    figsize=(12,6),
    color='purple'
)

plt.title('Highest Scoring IPL Venues')
plt.xlabel('Average Match Runs')
plt.ylabel('Venue')

for bar in ax.patches:
    plt.text(
        bar.get_width() + 2,
        bar.get_y() + bar.get_height()/2,
        f'{bar.get_width():.1f}',
        va='center'
    )

plt.tight_layout()
plt.savefig(
    'Images/Highest Scoring IPL Venues.png',
    dpi=500,
    bbox_inches='tight'
)
plt.show()

In [ ]:
matches.columns

In [ ]:
closest_wins = (
    matches[matches['result'] == 'runs']
    .sort_values('result_margin')
)

closest_wins[
    ['season','team1','team2','winner','result_margin']
].head(10)

In [ ]:
largest_wins = (
    matches[matches['result'] == 'runs']
    .sort_values('result_margin', ascending=False)
)

largest_wins[
    ['season','team1','team2','winner','result_margin']
].head(10)

In [ ]:
super_over_matches = matches[matches['super_over'] == 'Y']

print("Total Super Over Matches:", len(super_over_matches))

super_over_matches[
    ['season','team1','team2','winner']
].head(20)

In [ ]:
top_closest = (
    closest_wins.head(10)
    .sort_values('result_margin', ascending=False)
)

labels = (
    top_closest['team1']
    + " vs "
    + top_closest['team2']
)

plt.figure(figsize=(12,6))

bars = plt.barh(
    y=labels,
    width=top_closest['result_margin'],
    color='red'
)

plt.title('Closest IPL Matches')
plt.xlabel('Winning Margin (Runs)')
plt.ylabel('Match')

for bar in bars:
    plt.text(
        bar.get_width()+0.1,
        bar.get_y()+bar.get_height()/2,
        str(int(bar.get_width())),
        va='center'
    )

plt.tight_layout()
plt.savefig(
    'Images/closest IPL Matches.png',
    dpi=500,
    bbox_inches='tight'
)
plt.show()

In [ ]:
top_large = largest_wins.head(10).sort_values('result_margin')

top_large['winner_label'] = top_large['winner']

ax = top_large.plot(
    x='winner_label',
    y='result_margin',
    kind='barh',
    figsize=(12,6),
    color='darkblue'
)

plt.title('Largest Victories in IPL History')
plt.xlabel('Winning Margin (Runs)')
plt.ylabel('Winner')

for bar in ax.patches:
    plt.text(
        bar.get_width()+1,
        bar.get_y()+bar.get_height()/2,
        str(int(bar.get_width())),
        va='center'
    )

plt.tight_layout()
plt.savefig(
    'Images/Largest Victories in IPL History.png',
    dpi=500,
    bbox_inches='tight'
)
plt.show()

# Final Conclusions

## Key Findings

- Highest Team Total: Sunrisers Hyderabad (287)
- Highest Individual Score: Chris Gayle (175)
- Most Runs: Virat Kohli
- Most Wickets: YS Chahal
- Highest Scoring Venue: Visakhapatnam
- Closest Winning Margin: 1 Run

## Conclusion

IPL success depends on a combination of team quality, player consistency, venue conditions, and match strategy. While factors such as the toss may have some influence, long-term success is primarily driven by strong batting, effective bowling, and consistent team performance.